# mdlARC: ARC Prize 2026 ARC-AGI-2 submission

Uses the **final epoch-25 checkpoint** from
`vast-54994367-public-v3-b112-e25-bw0` (public originals, variations,
and ARC2 public-evaluation v3 compositions).

1. Attach the competition data and a Kaggle dataset containing `pretrain__tiny.pt`.
2. Choose **L4 × 4** and turn **Internet off**. This code uses **only GPU 0**.
3. Attach a compatible FlashAttention 2 wheel, or select `ATTENTION_BACKEND="native"`.
4. First run with `SMOKE_TEST=True`. Then restart, set it to **False**, and Run All.
5. **Save Version → Save & Run All** with `SMOKE_TEST=False`. Submit that version's
   `/kaggle/working/submission.json` through **Submit to Competition**.

The notebook contains its own mdlARC source; no source-code dataset is needed.
It finetunes **all parameters** on the supplied train pairs, then generates two
predictions for each runtime test input. It never reads test solutions.

Full instructions: `kaggle/README.md` in the local checkout.
Current rules: [Kaggle code requirements](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-2/overview/code-requirements).
Kaggle currently allows up to 12 hours and requires Internet off.
The full L4 run has to be timed in Kaggle; smoke mode checks execution, not accuracy.


In [ ]:
# SETTINGS — normally only change the checkpoint/wheel paths and SMOKE_TEST.
import os
import sys
from pathlib import Path

if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    if sys.modules["torch"].cuda.device_count() != 1:
        raise RuntimeError("Restart the session before running this notebook to select GPU 0.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

CHECKPOINT_PATH = None  # Auto-find pretrain__tiny.pt / tiny.pt, or paste its exact /kaggle/input/... path.
TEST_CHALLENGES = None  # Auto-find only the competition's arc-agi_test_challenges.json.
ATTENTION_BACKEND = "flash"  # "flash" = FlashAttention 2; "native" = PyTorch attention (no wheel).
FLASH_ATTN_WHEEL = None  # Exact /kaggle/input/.../flash_attn....whl path; None tries an installed package first.
ENABLE_COMPILE = True
SMOKE_TEST = False  # True: 2 tasks, 2 epochs, 2 augs; writes smoke_submission.json only.

EPOCHS = 330
TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch: 8 × 4 = 32 on one GPU.
EVAL_BATCH_SIZE = 8
TRAIN_AUGMENTS = 80
EVAL_AUGMENTS = 80
NUM_WORKERS = 2
NORMUON_LR = 0.0003
ADAMW_LR = 0.003
MAX_FINETUNE_HOURS = 8.0  # Abort early if steady-state ETA exceeds this; reserve time for decoding.

if ATTENTION_BACKEND not in {"flash", "native"}:
    raise ValueError('ATTENTION_BACKEND must be "flash" or "native".')
if int(os.environ.get("WORLD_SIZE", "1")) != 1 or "LOCAL_RANK" in os.environ:
    raise RuntimeError("Use the notebook directly in a fresh session, without torchrun/DDP.")
if not ENABLE_COMPILE:
    os.environ["TORCH_COMPILE_DISABLE"] = "1"
else:
    os.environ.pop("TORCH_COMPILE_DISABLE", None)
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
os.chdir(WORKING)
if (WORKING / "submission.json").exists():
    (WORKING / "submission.json").replace(WORKING / "previous_submission.json")
# Prevent any inherited remote-backup environment from affecting local saves.
os.environ.pop("MDLARC_CHECKPOINT_QUEUE", None)
os.environ.pop("MDLARC_RUN_DIR", None)
print(f"Mode: {'SMOKE TEST — do not submit' if SMOKE_TEST else 'FULL SUBMISSION'}")


## Environment and attention wheel

Run this cell before choosing a wheel. It prints the actual environment, rather
than assuming the four-month-old wheel still works. The wheel needs compatible
Python, PyTorch, CUDA, Linux and C++ ABI. L4 uses FlashAttention **2**, not the
Hopper-only FlashAttention 3 package. No packages are fetched from the internet.

If `flash` fails, attach a compatible wheel and restart, or explicitly select
`native`. Native attention preserves the prefix attention mask but can use more
memory/time. Reduce the microbatch if needed; keep the effective batch at 32
by increasing accumulation (for example 4 × 8).


In [ ]:
import importlib.util
import platform
import subprocess
import torch

print("Python:", sys.version)
print("Platform:", platform.platform(), platform.machine())
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
print("C++ ABI:", getattr(torch._C, "_GLIBCXX_USE_CXX11_ABI", "unknown"))
if not torch.cuda.is_available():
    raise RuntimeError("Select GPU L4 × 4 in Kaggle Settings, then restart the session.")
if torch.cuda.device_count() != 1:
    raise RuntimeError("GPU selection was applied too late. Restart, then Run All.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This optimizer needs a CUDA GPU with bfloat16 support; choose L4.")
print("Using:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(["nvidia-smi"], check=True)
for package in ("numpy", "numba"):
    if importlib.util.find_spec(package) is None:
        raise RuntimeError(f"Missing {package}; attach an offline wheel. Do not enable Internet for submission.")

flash_probe = [sys.executable, "-c", "from flash_attn import flash_attn_varlen_func; import flash_attn; print(flash_attn.__version__)"]
if ATTENTION_BACKEND == "flash":
    probe = subprocess.run(flash_probe, text=True, capture_output=True)
    if FLASH_ATTN_WHEEL is not None or probe.returncode:
        if FLASH_ATTN_WHEEL is None:
            tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
            wheels = sorted(p for p in Path("/kaggle/input").rglob("*.whl")
                            if p.name.startswith("flash_attn-") and tag in p.name)
            if len(wheels) != 1:
                raise RuntimeError(
                    f"No working FlashAttention import; found {len(wheels)} wheels for {tag}. "
                    "Attach a compatible wheel and set FLASH_ATTN_WHEEL, or set ATTENTION_BACKEND='native'.\n"
                    + probe.stderr[-3000:]
                )
            FLASH_ATTN_WHEEL = wheels[0]
        wheel = Path(FLASH_ATTN_WHEEL)
        if not wheel.is_file() or wheel.suffix != ".whl":
            raise FileNotFoundError(f"Offline wheel not found: {wheel}")
        print("Installing offline wheel:", wheel)
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                        "--force-reinstall", str(wheel)], check=True)
        probe = subprocess.run(flash_probe, text=True, capture_output=True)
    if probe.returncode:
        raise RuntimeError("FlashAttention wheel is incompatible with this environment.\n" + probe.stderr[-5000:])
    print("FlashAttention:", probe.stdout.strip())
else:
    print("Using native PyTorch attention; no FlashAttention wheel will be used.")
import torch._dynamo
for option in ("cache_size_limit", "recompile_limit"):
    if hasattr(torch._dynamo.config, option):
        setattr(torch._dynamo.config, option, 64)


## Embedded source

This payload contains only the mdlARC runtime and submission helpers. It includes
the latent-width fix needed when loading the 3072-wide embeddings for fewer than
3072 Kaggle tasks. Its checksum is verified before extraction.


In [ ]:
import base64
import hashlib
import io
import zipfile

RUNTIME_SHA256 = "ce40071dce0abc90fcaa2cf6f9dad7d0ab1abbdeda93e98b0b054f1a879e568a"
RUNTIME_B64 = "..."
runtime_bytes = base64.b64decode(RUNTIME_B64)
if hashlib.sha256(runtime_bytes).hexdigest() != RUNTIME_SHA256:
    raise RuntimeError("Embedded runtime checksum failed; re-upload the original notebook.")
RUNTIME = WORKING / "mdlarc_runtime"
with zipfile.ZipFile(io.BytesIO(runtime_bytes)) as archive:
    for entry in archive.infolist():
        target = (RUNTIME / entry.filename).resolve()
        target.relative_to(RUNTIME.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(entry))
sys.path.insert(0, str(RUNTIME / "src"))
sys.path.insert(0, str(RUNTIME / "kaggle"))
del runtime_bytes, RUNTIME_B64

if ATTENTION_BACKEND == "native":
    # Ignore an incompatible installed extension as well as attached wheels.
    sys.modules["flash_attn"] = None
    sys.modules["flash_attn_interface"] = None
import build
import tinytransformer
import train
import submission_utils as ku
if ATTENTION_BACKEND == "native":
    tinytransformer.flash_attn_varlen_func = None
    tinytransformer.flash_attn_varlen_qkvpacked_func = None
    tinytransformer.flash_attn_varlen_kvpacked_func = None
elif tinytransformer.flash_attn_varlen_func is None:
    raise RuntimeError("FlashAttention was not imported into the notebook kernel. Restart and Run All.")
print("Runtime SHA-256:", RUNTIME_SHA256)


## Checkpoint and runtime challenges

The expected final checkpoint is 4,364,936,923 bytes. Its hash is checked so an
epoch-10/20 checkpoint or an unrelated old model cannot be selected accidentally.
The original large checkpoint stays on the CPU. Old optimizer state and 99,201
pretraining factor rows are discarded; fresh factors are created for the runtime
tasks. Shared learned latent tables and transformer weights are retained.

Public evaluation answers were included in pretraining, so a public evaluation
score from this checkpoint would measure training overlap. Submit on the supplied
**test** challenges; Kaggle replaces that file with its scoring tasks.


In [ ]:
import json
import gc
from time import perf_counter

CHECKPOINT_PATH = ku.find_checkpoint(Path("/kaggle/input"), CHECKPOINT_PATH)
TEST_CHALLENGES = ku.find_test_challenges(Path("/kaggle/input"), TEST_CHALLENGES)
print("Checkpoint:", CHECKPOINT_PATH)
print("Challenges:", TEST_CHALLENGES)
if CHECKPOINT_PATH.stat().st_size != ku.FINAL_CHECKPOINT_SIZE:
    raise ValueError("Checkpoint size differs from the final epoch-25 backup. Use pretrain__tiny.pt.")
checkpoint_hash = ku.sha256_file(CHECKPOINT_PATH)
if checkpoint_hash != ku.FINAL_CHECKPOINT_SHA256:
    raise ValueError("Checkpoint SHA-256 differs from the verified final epoch-25 backup.")
print("Checkpoint SHA-256 verified:", checkpoint_hash)

data_path = WORKING / "runtime_challenges.json"
challenges = ku.prepare_challenges(TEST_CHALLENGES, data_path, smoke_test=SMOKE_TEST)
print("Tasks:", len(challenges), "train pairs:", sum(len(t["train"]) for t in challenges.values()),
      "test inputs:", sum(len(t["test"]) for t in challenges.values()))
# mmap avoids eagerly copying multi-GB optimizer tensors into RAM.
# weights_only=False is needed for this trusted, hash-verified legacy checkpoint.
original = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False, mmap=True)
adapted = ku.adapt_checkpoint(original)
print("Saved epoch:", original.get("epoch"), "saved architecture:", original["config"])
if original.get("epoch") != 25:
    raise ValueError("Expected the final pretrain epoch 25.")
if adapted["augmentation_sampling_mode"] != "single_grid_prefix":
    raise ValueError("Unexpected training format for this checkpoint.")
del original
gc.collect()


## Full-model finetuning

Starts a **fresh optimizer and schedule**, rather than resuming pretraining.
Default effective batch is 32; accumulation averages four microbatches on GPU 0.
All model parameters are trainable, including shared embeddings and MTP heads.
The checkpoint determines width/depth/MTP/dropout; no 1B-era architecture is used.
Finetune learning rates come from the repository's ARC2 finetune template; they
are starting settings, not a tuned optimum for this checkpoint.


In [ ]:
SAVED_DEFAULTS = json.loads("{\"seed\": 42, \"optimizer\": \"normuon\", \"FA\": 2, \"normuon_momentum\": 0.95, \"normuon_momentum_min\": 0.85, \"normuon_warmup_steps\": 0, \"normuon_cooldown_steps\": 0, \"normuon_beta2\": 0.95, \"normuon_bucket_size\": 0, \"normuon_qkv_lr_multiplier\": 1.0, \"normuon_attention_lr_multiplier\": 1.0, \"normuon_fc_in_lr_multiplier\": 1.0, \"normuon_fc_out_lr_multiplier\": 1.0, \"normuon_example_value_latent_lr_multiplier\": 1.0, \"normuon_other_lr_multiplier\": 1.0, \"normuon_residual_output_lr_multiplier\": 1.0, \"adamw_decay_lr_multiplier\": 1.0, \"adamw_attention_lr_multiplier\": 1.0, \"adamw_token_embedding_lr_multiplier\": 1.0, \"adamw_task_factor_lr_multiplier\": 1.0, \"adamw_example_value_latent_lr_multiplier\": 1.0, \"adamw_no_decay_lr_multiplier\": 1.0, \"weight_decay\": 0.1, \"attention_weight_decay\": 0.01, \"token_embedding_weight_decay\": 0.01, \"task_embedding_weight_decay\": 0.01, \"grad_clip\": 1.0, \"warmup_pct\": 0.005, \"resume_warmup_pct\": 0.0, \"wsd_decay_start_pct\": 0.8, \"lr_floor\": 0.0001, \"mtp_decay\": [0.8, 0.9, 1.0], \"augmentation_sampling_mode\": \"single_grid_prefix\", \"inference_temperature\": null, \"inference_top_k\": null}")
save_path = WORKING / "runs" / ("smoke_finetuned.pt" if SMOKE_TEST else "kaggle_finetuned.pt")
cfg = ku.finetune_config(
    SAVED_DEFAULTS, data_path=data_path, save_path=save_path,
    epochs=2 if SMOKE_TEST else EPOCHS,
    batch_size=1 if SMOKE_TEST else TRAIN_BATCH_SIZE,
    accumulation=1 if SMOKE_TEST else GRADIENT_ACCUMULATION_STEPS,
    max_augments=2 if SMOKE_TEST else TRAIN_AUGMENTS,
    num_workers=0 if SMOKE_TEST else NUM_WORKERS,
    normuon_lr=NORMUON_LR, adamw_lr=ADAMW_LR,
)
model, dataset, dataloader, device, _ = build.build_model_and_data(cfg, checkpoint=adapted)
if len(dataloader) < cfg.gradient_accumulation_steps:
    raise ValueError("Too few minibatches for the effective batch. Lower batch size/accumulation.")
if len(dataset.task_ids) != len(challenges) or len(dataset.embedding_task_ids) != len(challenges):
    raise RuntimeError("Unexpected task-factor mapping for runtime tasks.")
if model.config.num_task_embedding_latents != adapted["config"]["num_task_embedding_latents"]:
    raise RuntimeError("Shared task latent width changed.")
model.requires_grad_(True)
model._loaded_checkpoint = None  # Release old CPU weights; optimizer starts fresh.
del adapted
gc.collect()
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
factors = model.task_factor_embedding.weight.numel()
print(f"Trainable: {trainable:,} / {total:,} (100%); task factors: {factors:,}")
print("Effective batch:", cfg.batch_size * cfg.gradient_accumulation_steps)
print("CUDA forward/backward preflight loss:", ku.check_backward(model, dataloader, device))
torch.cuda.empty_cache()

epoch_times = []
real_train_one_epoch = train.train_one_epoch
def timed_train_one_epoch(*args, **kwargs):
    start = perf_counter()
    result = real_train_one_epoch(*args, **kwargs)
    torch.cuda.synchronize()
    elapsed = perf_counter() - start
    epoch_times.append(elapsed)
    # Skip the first epoch, which includes compilation warmup.
    if len(epoch_times) >= 3:
        seconds_per_epoch = sum(epoch_times[1:]) / (len(epoch_times) - 1)
        estimated_hours = (epoch_times[0] + seconds_per_epoch * (cfg.epochs - 1)) / 3600
        if len(epoch_times) in (3, 10, 25, 50, 100, 200):
            print(f"Estimated full finetune: {estimated_hours:.2f} hours; decoding is additional.")
        if not SMOKE_TEST and estimated_hours > MAX_FINETUNE_HOURS:
            raise RuntimeError(
                f"Finetune ETA {estimated_hours:.2f}h exceeds {MAX_FINETUNE_HOURS}h budget. "
                "Restart and lower EPOCHS, or try a compatible flash wheel / ENABLE_COMPILE=True. "
                "Leave room for decoding under the competition's 12h limit."
            )
    return result

train.train_one_epoch = timed_train_one_epoch
started = perf_counter()
try:
    train.train_model(cfg, model=model, dataloader=dataloader, dataset=dataset,
                      device=device, data_path=data_path)
finally:
    train.train_one_epoch = real_train_one_epoch
print(f"Finetune completed in {(perf_counter() - started) / 3600:.3f} hours.")
if not save_path.is_file():
    raise RuntimeError("Finetuning did not save a checkpoint.")
del model, dataset, dataloader
gc.collect()
torch.cuda.empty_cache()
torch._dynamo.reset()


## Decode and write the submission

Uses ordinary greedy decoding, 80 test-time augmentations, and the repository's
AAIVR aggregation to select two attempts. Both finetuning and decoding use the
same runtime challenge file. A malformed decoded grid is replaced with the other
valid attempt, or the corresponding input grid if neither attempt is valid;
replacements are reported. Missing tasks or test inputs cause an error.


In [ ]:
from argparse import Namespace
import evaluate

eval_cfg = Namespace(**vars(cfg))
eval_cfg.checkpoint_mode = None
eval_cfg.extend_checkpoint_task_ids = False
eval_cfg.inherit_checkpoint_rearc_data_path = False
eval_cfg.reset_training_state = False
eval_cfg.distributed_inference = False
eval_cfg.inference_speculative = False
eval_cfg.mode = "full-decode"
started = perf_counter()
run_name, eval_results, raw_submission = evaluate.run_evaluation(
    eval_cfg, run_name="smoke_decode" if SMOKE_TEST else "kaggle_decode",
    max_augments=2 if SMOKE_TEST else EVAL_AUGMENTS,
    data_path=data_path, checkpoint_path=save_path,
    batch_size=2 if SMOKE_TEST else EVAL_BATCH_SIZE, splits=["test"], task_ids=None,
    timing_path=WORKING / "runs" / "timing.txt",
)
output = WORKING / ("smoke_submission.json" if SMOKE_TEST else "submission.json")
repairs = ku.finalize_submission(raw_submission, output, challenges)
if repairs:
    print(f"Replaced {len(repairs)} malformed attempts:", repairs[:20])
print(f"Decode completed in {(perf_counter() - started) / 3600:.3f} hours.")
del eval_results
gc.collect()

summary = {
    "smoke_test": SMOKE_TEST, "checkpoint_sha256": checkpoint_hash,
    "runtime_sha256": RUNTIME_SHA256, "attention_backend": ATTENTION_BACKEND,
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "epochs": cfg.epochs, "effective_batch_size": cfg.batch_size * cfg.gradient_accumulation_steps,
    "train_augments": cfg.max_augments, "eval_augments": 2 if SMOKE_TEST else EVAL_AUGMENTS,
    "tasks": len(challenges), "test_inputs": sum(len(t["test"]) for t in challenges.values()),
    "malformed_attempt_replacements": repairs,
}
(WORKING / "submission_run_info.json").write_text(json.dumps(summary, indent=2))
if SMOKE_TEST:
    print("SMOKE TEST PASSED. Restart, set SMOKE_TEST=False, then Save & Run All. Do not submit smoke output.")
else:
    # Check against the complete competition input again, independently of the cached dict.
    full_tasks = json.loads(TEST_CHALLENGES.read_text())
    ku.validate_submission(json.loads(output.read_text()), full_tasks)
    print(f"READY: {output} ({output.stat().st_size:,} bytes), all runtime test inputs covered.")
    print("Submit submission.json from this successfully saved notebook version.")
